# Комиссии 4 клиентов из озера (DOG_OPER)

SQL коллеги: `ods.scd1_z_R2_IP_DOG_OPER` + `VID_COMISS` + `r2_ip_merchants` + `client`.

Клиенты: `413636181589`, `508492576892`, `818953761556`, `509630192857`.
Фильтр даты: `o.c_date_create > '2026-03-31'`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

AGR_IDS = [
    413636181589,
    508492576892,
    818953761556,
    509630192857,
]
agr_ids_sql = ', '.join(str(x) for x in AGR_IDS)
print('AGR_IDS:', AGR_IDS)
print('DATA_DIR:', DATA_DIR)

In [ ]:
if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp connection')

In [ ]:
# SQL коллеги (имена таблиц как на фото), без limit, на 4 agr_id
sql_commissions = f"""
select distinct
  m.c_cl_org cft_id,
  cl.c_name name_org,
  m.id id_agreement,
  m.c_name_in_pr agreement_num,
  m.c_date_begin,
  vc.id,
  vc.c_name commis_type,
  o.c_date_create,
  o.c_pay_summ,
  o.c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_R2_VID_COMISS vc on vc.id = o.c_vid_comiss
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and m.id in ({agr_ids_sql})
  and o.c_date_create > '2026-03-31'
order by o.c_date_create desc
"""

print(sql_commissions)

with imp:
    imp.execute('set MEM_LIMIT=8g')
    raw_df = imp.fetch(sql_commissions)

if raw_df is None:
    raw_df = pd.DataFrame()

print(f'raw rows: {len(raw_df):,}')
display(raw_df.head(50))

## Запасной SQL (lower-case имена таблиц)

Запускай **только если** предыдущая ячейка упала с «table not found».

In [ ]:
# Запасной вариант: lower-case имена таблиц (джойны/фильтры те же)
RUN_LOWERCASE_FALLBACK = False  # поставь True, если основной SQL не нашёл таблицы

if RUN_LOWERCASE_FALLBACK:
    sql_commissions_lc = f"""
    select distinct
      m.c_cl_org cft_id,
      cl.c_name name_org,
      m.id id_agreement,
      m.c_name_in_pr agreement_num,
      m.c_date_begin,
      vc.id,
      vc.c_name commis_type,
      o.c_date_create,
      o.c_pay_summ,
      o.c_calc_summ
    from ods.scd1_z_r2_ip_dog_oper o
    join ods.scd1_z_r2_vid_comiss vc on vc.id = o.c_vid_comiss
    join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
    join ods.scd1_z_client cl on m.c_cl_org = cl.id
    where o.c_parent_class = 'R2_IP_MERCHANTS'
      and cl.class_id = 'CL_ORG'
      and m.id in ({agr_ids_sql})
      and o.c_date_create > '2026-03-31'
    order by o.c_date_create desc
    """
    print(sql_commissions_lc)
    with imp:
        imp.execute('set MEM_LIMIT=8g')
        raw_df = imp.fetch(sql_commissions_lc)
    if raw_df is None:
        raw_df = pd.DataFrame()
    print(f'raw rows (lowercase): {len(raw_df):,}')
    display(raw_df.head(50))
else:
    print('SKIP lowercase fallback (RUN_LOWERCASE_FALLBACK=False)')

In [ ]:
# QC: строки на договор + пустые клиенты
if raw_df.empty:
    print('WARN: raw_df пустой — проверь SQL / доступ к таблицам / наличие операций')
else:
    work = raw_df.copy()
    work['id_agreement'] = work['id_agreement'].astype(str).str.replace(r'\.0$', '', regex=True)

    cnt = (
        work.groupby('id_agreement', as_index=False)
        .size()
        .rename(columns={'size': 'rows'})
        .sort_values('id_agreement')
    )
    print('Строк на id_agreement:')
    display(cnt)

    expected = {str(x) for x in AGR_IDS}
    found = set(cnt['id_agreement'].astype(str))
    missing = sorted(expected - found)
    if missing:
        print('Нет операций после 2026-03-31:', missing)
    else:
        print('Все 4 agr_id присутствуют в raw')

    empty_name = work['name_org'].isna() | (work['name_org'].astype(str).str.strip() == '')
    print(f'Строк с пустым name_org: {int(empty_name.sum())}')

In [ ]:
# Сводка: id_agreement × месяц → sum(c_pay_summ), sum(c_calc_summ)
MONTH_COL_RU = {
    '2026-04': 'апрель',
    '2026-05': 'май',
    '2026-06': 'июнь',
}

if raw_df.empty:
    by_month_long = pd.DataFrame()
    by_month_wide = pd.DataFrame()
    print('Нет данных для агрегации')
else:
    agg_src = raw_df.copy()
    agg_src['id_agreement'] = agg_src['id_agreement'].astype(str).str.replace(r'\.0$', '', regex=True)
    agg_src['c_date_create'] = pd.to_datetime(agg_src['c_date_create'], errors='coerce')
    agg_src['month'] = agg_src['c_date_create'].dt.strftime('%Y-%m')
    agg_src['month_ru'] = agg_src['month'].map(MONTH_COL_RU)
    agg_src['c_pay_summ'] = pd.to_numeric(agg_src['c_pay_summ'], errors='coerce')
    agg_src['c_calc_summ'] = pd.to_numeric(agg_src['c_calc_summ'], errors='coerce')

    attrs = (
        agg_src.sort_values(['id_agreement', 'c_date_create'])
        .groupby('id_agreement', as_index=False)
        .agg(
            cft_id=('cft_id', 'first'),
            name_org=('name_org', 'first'),
            agreement_num=('agreement_num', 'first'),
            c_date_begin=('c_date_begin', 'first'),
        )
    )

    by_month_long = (
        agg_src.groupby(['id_agreement', 'month', 'month_ru'], as_index=False)
        .agg(
            rows=('c_date_create', 'size'),
            c_pay_summ=('c_pay_summ', 'sum'),
            c_calc_summ=('c_calc_summ', 'sum'),
        )
        .sort_values(['id_agreement', 'month'])
        .reset_index(drop=True)
    )

    pay_wide = (
        by_month_long.pivot(index='id_agreement', columns='month_ru', values='c_pay_summ')
        .reindex(columns=['апрель', 'май', 'июнь'])
        .add_prefix('pay_')
    )
    calc_wide = (
        by_month_long.pivot(index='id_agreement', columns='month_ru', values='c_calc_summ')
        .reindex(columns=['апрель', 'май', 'июнь'])
        .add_prefix('calc_')
    )

    by_month_wide = (
        attrs.set_index('id_agreement')
        .join(pay_wide, how='left')
        .join(calc_wide, how='left')
        .reset_index()
    )

    # порядок как в AGR_IDS + пустые строки для отсутствующих
    order = {str(a): i for i, a in enumerate(AGR_IDS)}
    missing = [str(a) for a in AGR_IDS if str(a) not in set(by_month_wide['id_agreement'].astype(str))]
    if missing:
        empty = pd.DataFrame([{'id_agreement': a} for a in missing])
        by_month_wide = pd.concat([by_month_wide, empty], ignore_index=True)

    by_month_wide['_ord'] = by_month_wide['id_agreement'].astype(str).map(order)
    by_month_wide = by_month_wide.sort_values('_ord').drop(columns=['_ord']).reset_index(drop=True)

    print('by_month_long:')
    display(by_month_long)
    print('by_month_wide (pay_* / calc_* = апрель/май/июнь):')
    display(by_month_wide)

In [ ]:
out_raw = DATA_DIR / 'lake_dog_oper_commissions_4agr_raw.xlsx'
out_month = DATA_DIR / 'lake_dog_oper_commissions_4agr_by_month.xlsx'

raw_df.to_excel(out_raw, index=False)

with pd.ExcelWriter(out_month, engine='openpyxl') as writer:
    by_month_wide.to_excel(writer, sheet_name='by_month_wide', index=False)
    by_month_long.to_excel(writer, sheet_name='by_month_long', index=False)

print('Saved:', out_raw)
print('Saved:', out_month)
print(f'raw rows={len(raw_df):,} | by_month_wide={len(by_month_wide):,} | by_month_long={len(by_month_long):,}')